In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum as _sum,
    avg,
    max as _max,
    min as _min,
    desc,
    to_date,
    window,
    rand
)

import os
import time

spark = SparkSession.builder \
    .appName("BigDataScalableAnalysis") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

start_time = time.time()

large_df = spark.range(0, 1000000)

large_df = large_df.withColumn(
    "product",
    (col("id") % 5).cast("string")
)

large_df = large_df.withColumn(
    "price",
    (rand() * 1000).cast("int")
)

large_df = large_df.withColumn(
    "quantity",
    ((rand() * 10) + 1).cast("int")
)

large_df = large_df.withColumn(
    "region",
    ((col("id") % 4) + 1).cast("string")
)

large_df = large_df.withColumn(
    "customer_id",
    ((col("id") % 50000) + 1).cast("string")
)

large_df = large_df.withColumn(
    "transaction_date",
    to_date(
        ((col("id") % 28) + 1).cast("string"),
        "d"
    )
)

large_df = large_df.replace(
    ["0", "1", "2", "3", "4"],
    ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones"],
    "product"
)

large_df = large_df.replace(
    ["1", "2", "3", "4"],
    ["East", "West", "North", "South"],
    "region"
)

df_clean = large_df.dropna()

df_clean = df_clean.withColumn(
    "revenue",
    col("price") * col("quantity")
)

df_clean.cache()

total_records = df_clean.count()

print("Total Records:", total_records)

df_clean.printSchema()

revenue_summary = df_clean.agg(
    _sum("revenue").alias("total_revenue"),
    avg("revenue").alias("average_revenue"),
    _max("revenue").alias("maximum_revenue"),
    _min("revenue").alias("minimum_revenue")
)

revenue_summary.show()

top_products = df_clean.groupBy("product") \
    .agg(
        _sum("revenue").alias("total_revenue"),
        count("product").alias("total_sales")
    ) \
    .orderBy(desc("total_revenue"))

top_products.show()

region_sales = df_clean.groupBy("region") \
    .agg(
        _sum("revenue").alias("region_revenue"),
        avg("revenue").alias("avg_region_revenue")
    ) \
    .orderBy(desc("region_revenue"))

region_sales.show()

customer_stats = df_clean.groupBy("customer_id") \
    .agg(
        _sum("revenue").alias("total_spent"),
        count("customer_id").alias("purchase_count")
    ) \
    .orderBy(desc("total_spent"))

customer_stats.show(10)

daily_sales = df_clean.groupBy("transaction_date") \
    .agg(
        _sum("revenue").alias("daily_revenue")
    ) \
    .orderBy("transaction_date")

daily_sales.show()

weekly_trend = df_clean.groupBy(
    window(col("transaction_date"), "7 days")
).agg(
    _sum("revenue").alias("weekly_revenue")
).orderBy("window")

weekly_trend.show()

top_products.explain(True)

df_partitioned = df_clean.repartition(8, "region")

print("Number of Partitions:", df_partitioned.rdd.getNumPartitions())

output_path = "output/revenue_summary"

if not os.path.exists("output"):
    os.makedirs("output")

revenue_summary.write.mode("overwrite").csv(output_path)

end_time = time.time()

print("Execution Time:", round(end_time - start_time, 2), "seconds")

spark.stop()

Total Records: 1000000
root
 |-- id: long (nullable = false)
 |-- product: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- revenue: integer (nullable = true)

+-------------+---------------+---------------+---------------+
|total_revenue|average_revenue|maximum_revenue|minimum_revenue|
+-------------+---------------+---------------+---------------+
|   2743132619|    2743.132619|           9990|              0|
+-------------+---------------+---------------+---------------+

+----------+-------------+-----------+
|   product|total_revenue|total_sales|
+----------+-------------+-----------+
|   Monitor|    549309724|     200000|
|     Mouse|    549035380|     200000|
|    Laptop|    548611159|     200000|
|  Keyboard|    548579562|     200000|
|Headphones|    547596794|     200000|
+----------+--------